# D1 — Periodontitis Field-Spec Ablation

Cumulative-add ablation on the **patient_population** form (17 scalar fields). Each variant adds one spec layer back:

| Variant | Keeps | Isolates |
|---|---|---|
| `abl1_desc` | description only | floor |
| `abl2_desc_hints` | + hints | hints |
| `abl3_desc_hints_rules` | + rules | rules |
| `abl4_desc_hints_rules_examples` | + examples | examples |
| `abl5_full` | + source-grounding | = production |

**Offline** — base `schema_def` comes from `eval/studies/ablation/base_schemas/*.json` (pulled from the live DB). Nothing is read from or written to Supabase. Extraction uses the Anthropic key in `eval/.env`.

### Before you run
1. Select the **`topics`** conda env as the kernel (`/home/ubuntu/miniconda3/envs/topics`) — it has `dspy`, `litellm`, etc.
2. Run the cells top to bottom: **setup → dry-run → smoke (2 papers) → full (29×5) → score**.
3. The smoke test writes 2-paper CSVs; the full run overwrites them with all 29. Score only after the full run.

## 1 · Setup — paths, env, dependency check

In [ ]:
import os, sys

os.chdir('/home/ubuntu/evistream')
for p in ('/home/ubuntu/evistream', '/home/ubuntu/evistream/backend', '/home/ubuntu/evistream/eval'):
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ.setdefault('USE_RUNTIME_BUILDERS', 'true')  # build DSPy classes from schema_def at runtime

from dotenv import load_dotenv
load_dotenv('eval/.env')         # ANTHROPIC_API_KEY, LLM_JUDGE_MODEL
load_dotenv('backend/.env')      # any extra config (optional)

import dspy, litellm  # noqa: F401
print('python      :', sys.executable)
print('dspy        :', dspy.__version__)
print('ANTHROPIC   :', 'set' if os.environ.get('ANTHROPIC_API_KEY') else 'MISSING')
assert os.environ.get('ANTHROPIC_API_KEY'), 'No ANTHROPIC_API_KEY — extraction will fail'
print('\nSetup OK.')

## 2 · Load base schema + dry-run (inspect what each variant strips)
No extraction, no cost — just confirms V1→V5 remove the right layers.

In [ ]:
from eval.studies.ablation.run_perio_ablation import (
    _load_base_schema_def, _make_config, _output_field_order,
    _load_papers, _run_one_variant,
)
from eval.studies.ablation.config import VARIANTS, strip_list_for

FORM = 'patient_population'
base = _load_base_schema_def(FORM)
field_order = _output_field_order(base)
print(f"base schema : {base['schema_name']}  ({len(field_order)} fields)\n")

for v in VARIANTS:
    _, vdef = _make_config(base, v)
    f0 = vdef['signatures'][0]['output_fields'][0]
    kept = [k for k in ('hints', 'rules', 'examples') if f0.get(k)]
    print(f"{v.name:<34} strips={str(strip_list_for(v) or '(none)'):<48} "
          f"e.g. {f0['name']} keeps {kept or ['description only']}, sg={f0.get('source_grounded')}")

## 3 · Smoke test — 2 papers × 5 variants (10 extractions)
Sanity-check the pipeline end-to-end before the full run. Writes 2-paper CSVs (overwritten in step 4).

In [ ]:
papers = _load_papers(2)
for v in VARIANTS:
    await _run_one_variant(base, v, papers, field_order, concurrency=8)
print('\nSmoke test done. Spot-check a CSV below, then run step 4.')

In [ ]:
import pandas as pd
pd.read_csv('eval/sheets/ai sheets/desc_only/periodontitis/claude/patient_population_abl5_full_long.csv').head()

## 4 · Full run — 29 papers × 5 variants (145 extractions)
This is the real run. Takes a while and costs Anthropic tokens. Progress prints per variant.

In [ ]:
papers = _load_papers(None)   # all 29
print(f'Running {len(VARIANTS)} variants × {len(papers)} papers ...\n')
for v in VARIANTS:
    await _run_one_variant(base, v, papers, field_order, concurrency=8)
print('\nAll variants extracted → eval/sheets/ai sheets/desc_only/periodontitis/claude/')

## 5 · Score each variant vs ground truth
Reuses the exact periodontitis harness (`run_form` + `PERIODONTITIS_REGISTRY` + `periodontitis.xlsx`). The headline is macro/micro F1 climbing V1→V5.

In [ ]:
from eval.studies.ablation.score_perio_ablation import _score_variant, _write_summary

summaries = []
for v in VARIANTS:
    s = _score_variant(FORM, v.name, use_llm=True)
    if s:
        summaries.append(s)
        print(f"  scored {v.name:<34} macro_f1={s.get('macro_f1')}  micro_f1={s.get('micro_f1')}")

out = _write_summary(FORM, summaries)
print(f'\nSummary sheet → {out}')

import pandas as pd
cols = ['variant', 'n_scored_fields', 'macro_f1', 'micro_f1', 'macro_precision', 'macro_recall']
df = pd.DataFrame(summaries)
df[[c for c in cols if c in df.columns]]

## 6 · (optional) Plot the F1 climb V1→V5

In [ ]:
import matplotlib.pyplot as plt

labels = ['desc', '+hints', '+rules', '+examples', '+src-grnd']
f1 = [s.get('macro_f1') for s in summaries]
plt.figure(figsize=(7, 4))
plt.plot(labels, f1, marker='o')
plt.ylabel('macro F1'); plt.title('Field-spec ablation — patient_population (periodontitis)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()